# Renombrar transcripciones a IDs cortos

Este notebook  genera copias con nombres cortos y legibles en una carpeta nueva
`Transcriptions procesadas/`.

Formato del nuevo nombre: `audio{h|i}{n}-{primeros 8 caracteres del UUID}-{n_hablantes}spk`

- `h` = humano, `i` = IA
- `n` = número secuencial dentro de su grupo (orden alfabético del UUID original, estable)
- `n_hablantes` = `n_speakers_detected` que ya trae el JSON (1, 2 o 3), útil para ver de un
  vistazo qué audios necesitan revisión especial (los de 1 y 3)

Ejemplo: `0445c357-e465-49ae-8067-bfa91d11b532.json` (humano, 1er archivo, 2 hablantes)
pasa a llamarse `audioh1-0445c357-2spk.json`.

In [18]:
import json
import shutil
from pathlib import Path
import csv

# Ajusta esta ruta si el notebook no corre desde la raíz del repo
REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "Transcriptions").exists():
    REPO_ROOT = REPO_ROOT.parent  # por si se corre desde /notebooks

SRC_DIR = REPO_ROOT / "Transcriptions"
DST_DIR = REPO_ROOT / "Transcriptions procesadas" / "Transcriptions_renombradas"

print("Origen: ", SRC_DIR, "| existe:", SRC_DIR.exists())
print("Destino:", DST_DIR, "| existe:", DST_DIR.exists())

Origen:  C:\Users\andre\Desktop\Personal\Pruebas tecnicas\crecere\prueba_crecere\Transcriptions | existe: True
Destino: C:\Users\andre\Desktop\Personal\Pruebas tecnicas\crecere\prueba_crecere\Transcriptions procesadas\Transcriptions_renombradas | existe: False


In [19]:
# Carpetas de origen -> prefijo corto
GROUPS = [
    ("humanos", "h"),
    ("ia", "i"),
]

for _, prefix in GROUPS:
    (DST_DIR / ("humanos" if prefix == "h" else "ia")).mkdir(parents=True, exist_ok=True)

print("Carpetas de destino listas.")

Carpetas de destino listas.


In [20]:
mapeo = []  # filas para mapeo_ids.csv

for folder_name, prefix in GROUPS:
    src_folder = SRC_DIR / folder_name
    files = sorted(src_folder.glob("*.json"))  # orden alfabético del UUID = estable

    for i, filepath in enumerate(files, start=1):
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)

        original_id = data["file_id"]
        n_speakers = data.get("n_speakers_detected", "NA")
        short_id = original_id[:8]

        new_name = f"audio{prefix}{i}-{short_id}-{n_speakers}spk"
        new_filename = f"{new_name}.json"

        # Copiamos el contenido y le agregamos el nuevo id como metadata adicional,
        # SIN borrar el file_id original (trazabilidad dentro del propio JSON también)
        data_copy = dict(data)
        data_copy["short_id"] = new_name
        data_copy["original_file_id"] = original_id

        dst_path = DST_DIR / folder_name / new_filename
        with open(dst_path, "w", encoding="utf-8") as f:
            json.dump(data_copy, f, ensure_ascii=False, indent=2)

        mapeo.append({
            "short_id": new_name,
            "original_file_id": original_id,
            "source": data.get("source", folder_name),
            "n_speakers_detected": n_speakers,
            "duration_sec": data.get("duration_sec"),
            "n_segments": len(data.get("segments", [])),
            "new_path": str(dst_path.relative_to(REPO_ROOT)),
            "original_path": str(filepath.relative_to(REPO_ROOT)),
        })

print(f"Procesados {len(mapeo)} archivos.")

Procesados 100 archivos.


In [21]:
# Guardar el mapeo (siempre poder volver del nombre corto al UUID original)
mapeo_path = DST_DIR / "mapeo_ids.csv"

fieldnames = [
    "short_id", "original_file_id", "source", "n_speakers_detected",
    "duration_sec", "n_segments", "new_path", "original_path",
]

with open(mapeo_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(mapeo)

print("Mapeo guardado en:", mapeo_path)
print(f"Total filas: {len(mapeo)}")

Mapeo guardado en: C:\Users\andre\Desktop\Personal\Pruebas tecnicas\crecere\prueba_crecere\Transcriptions procesadas\Transcriptions_renombradas\mapeo_ids.csv
Total filas: 100


In [22]:
# Verificación rápida: no se sobrescribió nada en la carpeta original,
# y la cantidad de archivos nuevos coincide con la original
n_original = sum(len(list((SRC_DIR / folder).glob("*.json"))) for folder, _ in GROUPS)
n_nuevos = sum(len(list((DST_DIR / folder).glob("*.json"))) for folder, _ in GROUPS)

print("Archivos originales:", n_original)
print("Archivos nuevos:     ", n_nuevos)
assert n_original == n_nuevos, "¡Descuadre! revisar antes de continuar"
print("OK: la carpeta original sigue intacta y la copia tiene la misma cantidad de archivos.")

Archivos originales: 100
Archivos nuevos:      100
OK: la carpeta original sigue intacta y la copia tiene la misma cantidad de archivos.


In [23]:
# Vista previa de algunos nombres nuevos, para confirmar visualmente el formato
for fila in mapeo[:5] + mapeo[-5:]:
    print(f"{fila['short_id']:22s} <- {fila['original_file_id']} ({fila['source']}, {fila['n_speakers_detected']} hablantes)")

audioh1-0445c357-2spk  <- 0445c357-e465-49ae-8067-bfa91d11b532 (humano, 2 hablantes)
audioh2-09115a4a-2spk  <- 09115a4a-ac50-4425-9dbc-b551a0b5342c (humano, 2 hablantes)
audioh3-0bc9a430-1spk  <- 0bc9a430-bdcd-44a7-a92d-e71dc33c8ad5 (humano, 1 hablantes)
audioh4-0c6ada7d-1spk  <- 0c6ada7d-d8c5-4840-ae79-0d8d573183dc (humano, 1 hablantes)
audioh5-0d2d37dd-2spk  <- 0d2d37dd-38a4-40dd-ac62-b430f722ed57 (humano, 2 hablantes)
audioi46-c8b57e01-2spk <- c8b57e01-57b6-4d85-9613-c786aef9f8a0 (ia, 2 hablantes)
audioi47-da02116f-2spk <- da02116f-5f55-4cc7-a800-373f4939fa01 (ia, 2 hablantes)
audioi48-edbc4154-1spk <- edbc4154-c6ce-4a77-b899-caf949e1815a (ia, 1 hablantes)
audioi49-f55b661d-2spk <- f55b661d-8a1b-4528-976e-d7b2f565b2ea (ia, 2 hablantes)
audioi50-fc9da4d5-2spk <- fc9da4d5-10dd-4aab-afb0-de918f081149 (ia, 2 hablantes)


## Eliminar audios sin contacto real

Estos dos audios detectaron 1 solo hablante porque en realidad **no hubo conversación**:
el usuario nunca contestó (uno es buzón de voz, el otro es la IA colgando tras no obtener
respuesta). No aportan información para comparar desempeño humano vs. IA, así que se
eliminan de la carpeta renombrada (la carpeta `Transcriptions` original NO se toca) y se
quitan también del `mapeo_ids.csv` para que quede consistente.

- `1c72dd55-acbb-4ebe-a4d7-446cc294a4a1` (humano) — buzón de voz, nadie contesta.
- `edbc4154-c6ce-4a77-b899-caf949e1815a` (ia) — la IA cuelga tras no obtener respuesta.

In [24]:
IDS_A_ELIMINAR = [
    "1c72dd55-acbb-4ebe-a4d7-446cc294a4a1",  # humano - buzón de voz, sin contacto
    "edbc4154-c6ce-4a77-b899-caf949e1815a",  # ia - cuelga sin respuesta
]

eliminados = []
mapeo_filtrado = []

for fila in mapeo:
    if fila["original_file_id"] in IDS_A_ELIMINAR:
        ruta = REPO_ROOT / fila["new_path"]
        if ruta.exists():
            ruta.unlink()
            eliminados.append(fila)
            print(f"Eliminado: {fila['short_id']} ({fila['original_file_id']})")
        else:
            print(f"Aviso: no se encontró el archivo para {fila['short_id']}, se quita del mapeo igual")
            eliminados.append(fila)
    else:
        mapeo_filtrado.append(fila)

print(f"\nTotal eliminados: {len(eliminados)} (esperado: {len(IDS_A_ELIMINAR)})")
assert len(eliminados) == len(IDS_A_ELIMINAR), "No se encontraron los dos IDs esperados, revisar"

mapeo = mapeo_filtrado  # actualizamos la lista en memoria para el resto del notebook

Eliminado: audioh6-1c72dd55-1spk (1c72dd55-acbb-4ebe-a4d7-446cc294a4a1)
Eliminado: audioi48-edbc4154-1spk (edbc4154-c6ce-4a77-b899-caf949e1815a)

Total eliminados: 2 (esperado: 2)


In [25]:
# Reescribir mapeo_ids.csv sin las filas eliminadas
with open(mapeo_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(mapeo)

print("mapeo_ids.csv actualizado. Filas restantes:", len(mapeo))

# Verificación final: la carpeta renombrada debe tener 98 archivos (100 - 2 excluidos)
n_final = sum(len(list((DST_DIR / folder).glob("*.json"))) for folder, _ in GROUPS)
print("Archivos finales en Transcriptions_renombradas:", n_final)
assert n_final == 98, "Se esperaban 98 archivos tras excluir los 2 casos sin contacto"

mapeo_ids.csv actualizado. Filas restantes: 98
Archivos finales en Transcriptions_renombradas: 98


### Despues de renombrar las transcripciones tenemos que averiguar sobre quien es SPEAKER1 y SPEAKER2.

#### Prompt A — 2/3 hablantes

Eres un analista que revisa transcripciones de llamadas de cobranza (call center) en español.
Cada llamada tiene hablantes etiquetados SPEAKER_00, SPEAKER_01 (y en algunos casos también
UNKNOWN). Estas etiquetas son arbitrarias y NO significan lo mismo entre llamadas distintas.

Tu tarea: para cada llamada, decide cuál etiqueta corresponde al AGENTE (quien llama en
nombre de una empresa de cobranza/banco) y cuál al DEUDOR/CLIENTE (la persona contactada).
Si aparece la etiqueta UNKNOWN, ignórala (es ruido residual, no un tercer hablante real).

Pistas típicas del agente: habla primero, se identifica ("le habla de...", "mi nombre es..."),
menciona una empresa/banco/"casa de cobranza", pregunta por el nombre del deudor, explica el
motivo de la llamada (obligación, deuda, pago).
Pistas típicas del deudor: responde preguntas, confirma su identidad, hace preguntas sobre su
deuda, da excusas o acepta/rechaza propuestas.

Si la transcripción no da suficiente evidencia para decidir con confianza, usa "incierto"
en vez de adivinar.

Responde ÚNICAMENTE con un JSON válido (sin texto antes ni después, sin markdown, sin
```json), con esta estructura exacta:

{
  "resultados": [
    {
      "short_id": "audioh2-09115a4a-2spk",
      "agente": "SPEAKER_01",
      "deudor": "SPEAKER_00",
      "confianza": "alta",
      "evidencia": "habla primero y se identifica"
    }
  ]
}

Donde:
- short_id es el identificador que aparece en el encabezado de cada llamada
- agente y deudor son "SPEAKER_00" o "SPEAKER_01" (o "incierto")
- confianza es "alta", "media" o "baja"
- evidencia es una frase MUY corta (máx 8 palabras)

Debe haber un objeto en "resultados" por cada llamada que te paso, ni más ni menos.

Aquí están las llamadas (primeros segmentos de cada una):

--- INICIO DE DATOS ---
[PEGAR BLOQUE]
--- FIN DE DATOS ---

#### Prompt B — 1 hablante

Eres un analista que revisa transcripciones de llamadas de cobranza en español. En esta
llamada, la diarización automática falló y atribuyó TODO el audio a un solo hablante, pero en
realidad es una conversación real entre un AGENTE de cobranza y un DEUDOR/cliente.

Tu tarea: recorre los segmentos EN ORDEN e infiere, solo por el contenido (quién pregunta vs.
quién responde, quién se identifica como empresa vs. quién confirma su propia identidad,
quién propone vs. quién acepta/rechaza), a qué rol pertenece cada segmento: "AGENTE" o
"DEUDOR". Si un segmento es ambiguo (ej. una respuesta corta como "sí", "bien"), infiere por
el contexto de los segmentos inmediatamente anteriores.

Responde ÚNICAMENTE con un JSON válido (sin texto antes ni después, sin markdown, sin
```json), con esta estructura exacta:

{
  "short_id": "audioh6-1c72dd55-1spk",
  "segmentos": [
    {"indice": 0, "rol": "AGENTE"},
    {"indice": 1, "rol": "DEUDOR"},
    {"indice": 2, "rol": "AGENTE"}
  ]
}

Debe haber un objeto en "segmentos" por cada segmento de la llamada, en el mismo orden en
que aparecen, usando el índice que te indico en cada uno (empezando en 0).

Aquí está la llamada completa:

--- INICIO DE DATOS ---
[PEGAR BLOQUE]
--- FIN DE DATOS ---
